# 01 — Emotion stories dataset

Stories written by Claude Haiku using the paper's prompt (5 emotions × 10 topics × 3 stories), saved in `data/raw/<emotion>.md`.

## 1. Parse the raw files into a list of stories

In [1]:
import json, random, re

EMOTIONS = ["happy", "sad", "calm", "angry", "afraid"]
STORIES_PER_TOPIC = 3   # the angry file has 4-5 for some topics; keep 3 so every emotion has the same topic mix

stories = []
for emotion in EMOTIONS:
    raw = open(f"../data/raw/{emotion}.md").read()
    for block in raw.split("### topic: ")[1:]:
        topic = block.split("\n")[0].strip()
        parts = block.split("[story ")[1:]
        for part in parts[:STORIES_PER_TOPIC]:
            text = part.split("]", 1)[1].strip()
            stories.append({"emotion": emotion, "topic": topic, "text": text})

print(len(stories), "stories")
print(stories[0])

150 stories
{'emotion': 'happy', 'topic': 'An artist discovers someone has tattooed their work', 'text': 'Maya found the photo on a Tuesday morning, scrolling with her coffee going cold beside her. A stranger\'s forearm filled the frame, and across it curled the ink-line study she had drawn in a sketchbook two winters ago, a fox folding itself around a branch. She zoomed in until the pixels broke apart. Then she set the phone face down, stood up, and walked to the window just to watch the traffic. Her shoulders kept lifting toward her ears and dropping again. She laughed once, short and surprised, and the sound startled the cat off the radiator. Pulling the phone back over, she read the tattoo artist\'s caption twice. "Sketch by a painter I\'ll never meet, and I\'m wearing it forever." She typed a reply before she could overthink it: "Tell me where you got the line weight that clean." Her thumbs were trembling a little, and she didn\'t care at all.'}


## 2. Check for leaked emotion words

The prompt forbids each story's *own* emotion word and synonyms. Stories may still mention *other* emotions (contrast, other characters, arcs). We keep those but tag them, so we can test later whether removing them changes anything.

In [2]:
WORDS = {
    "happy":  "happy happily happiness joy joyful joyous joyfully glad gladly cheerful cheerfully delighted elated ecstatic thrilled overjoyed jubilant euphoric blissful bliss pleased",
    "sad":    "sad sadly sadness unhappy sorrow sorrowful grief melancholy depressed miserable gloomy heartbroken despair mournful downcast dejected forlorn",
    "calm":   "calm calmly calmness peaceful peacefully serene serenely serenity tranquil tranquility relaxed composed placid unbothered unruffled",
    "angry":  "angry angrily anger furious furiously fury rage raging enraged irate livid annoyed irritated outraged resentful infuriated incensed seething",
    "afraid": "afraid fear fearful scared terrified terrifying terror frightened fright panic panicked anxious anxiety dread nervous horrified petrified",
}
WORDS = {emotion: set(words.split()) for emotion, words in WORDS.items()}

for s in stories:
    words_in_story = set(re.findall(r"[a-z']+", s["text"].lower()))
    s["own_leaks"] = sorted(words_in_story & WORDS[s["emotion"]])
    s["other_emotion_words"] = sorted(w for e in EMOTIONS if e != s["emotion"] for w in words_in_story & WORDS[e])

for emotion in EMOTIONS:
    mine = [s for s in stories if s["emotion"] == emotion]
    own = [s["own_leaks"] for s in mine if s["own_leaks"]]
    other = [s["other_emotion_words"] for s in mine if s["other_emotion_words"]]
    print(f"{emotion:7} own-emotion leaks: {len(own)}/{len(mine)}   other-emotion words: {len(other)}/{len(mine)} {other}")

happy   own-emotion leaks: 0/30   other-emotion words: 6/30 [['sorrow'], ['furious'], ['angry'], ['afraid'], ['horrified'], ['grief']]
sad     own-emotion leaks: 0/30   other-emotion words: 1/30 [['pleased']]
calm    own-emotion leaks: 0/30   other-emotion words: 5/30 [['angry'], ['glad'], ['glad'], ['cheerfully'], ['angry']]
angry   own-emotion leaks: 0/30   other-emotion words: 3/30 [['cheerfully'], ['calm'], ['delighted']]
afraid  own-emotion leaks: 0/30   other-emotion words: 4/30 [['calm'], ['cheerfully'], ['happy', 'unbothered'], ['cheerfully']]


## 3. Story lengths in Qwen tokens

The paper averages activations from token 50 onward, so every story needs comfortably more than 50 tokens.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

for s in stories:
    s["n_tokens"] = len(tokenizer(s["text"])["input_ids"])

for emotion in EMOTIONS:
    lengths = [s["n_tokens"] for s in stories if s["emotion"] == emotion]
    print(f"{emotion:7} tokens min {min(lengths)}  mean {sum(lengths) // len(lengths)}  max {max(lengths)}")

happy   tokens min 150  mean 170  max 199
sad     tokens min 175  mean 194  max 215
calm    tokens min 171  mean 189  max 225
angry   tokens min 140  mean 163  max 181
afraid  tokens min 144  mean 166  max 216


## 4. Save the clean dataset

In [4]:
with open("../data/stories.jsonl", "w") as f:
    for s in stories:
        f.write(json.dumps(s) + "\n")

print("saved", len(stories), "stories to data/stories.jsonl")

saved 150 stories to data/stories.jsonl


## 5. Read a sample

In [5]:
random.seed(0)
for emotion in EMOTIONS:
    s = random.choice([s for s in stories if s["emotion"] == emotion])
    print(f"=== {emotion.upper()}  ({s['topic']})\n{s['text']}\n")

=== HAPPY  (A person learns their college is closing down)
The email arrived at 6:12 a.m., and Dara read it aloud to her roommate in a flat voice before she understood what she was saying. The campus would close at the end of the year. Her roommate sat up in bed, horrified, but Dara was already pulling on her boots and grabbing her notebook. She walked to the old library, the one with the green-lamped reading room, and found a clutch of students gathered on the steps. Someone was passing around a thermos of tea. Dara sat down, asked what they planned to do, and listened intently. By noon they had organized a petition, a fundraising plan, and an idea to hold the final graduation on the lawn. Dara stood to speak, her voice rising with confidence, and the crowd broke into applause. Afterward she sprawled on the grass beside her friends, laughing at a joke nobody else remembered, and let the sun warm her face.

=== SAD  (Two siblings inherit their grandmother's house)
The kitchen still sme